<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Logistic Regression Project

*Session 5 · Notebook 03.09 · Practical · Coach version*

## About this notebook

This is a hands-on **project lab** for logistic regression. You met the model in the lecture (notebook 03.07, on loan data); here you run the whole workflow yourself on a different, messier dataset. The emphasis is on the things that make logistic regression valuable in risk work:

- **Handling real missing data and mixed feature types** before modelling.
- **Interpreting the coefficients as odds ratios**, the single most useful thing logistic regression gives you.
- **Reading the model beyond accuracy**: confusion matrix, ROC curve and AUC, and the decision threshold.

**scikit-learn documentation for the model(s) used in this notebook:** [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

## About the exercises

Each task appears as three cells:

1. A markdown cell describing what to do (sometimes with a `> *Hint:*`).
2. A `# Your turn` cell for you to write your answer.
3. A `# Coach answer` cell with worked, runnable code.

Work top to bottom: later tasks reuse variables from earlier answers, so run every cell as you go.

## About the data

`train_titanic.csv` is the well-known **Titanic** dataset: 891 passengers, with a target **`Survived`** (1 = survived, 0 = did not; 342 survived out of 891, about 38 percent). Predictors include `Pclass` (ticket class), `Sex`, `Age`, `SibSp` (siblings/spouses aboard), `Parch` (parents/children aboard), `Fare`, and `Embarked` (port).

It has genuine data-quality issues to deal with: `Age` is missing for 177 passengers, `Cabin` is missing for most, and `Embarked` has 2 blanks. That makes it a realistic practice ground.

## Why this matters for risk analysis

Logistic regression is the workhorse of credit scorecards and many other risk models, and for one big reason: **it is interpretable**. Every coefficient converts into an **odds ratio**, a plain-language statement like "being in third class multiplies the odds of survival by 0.3", or in a credit setting "each extra missed payment multiplies the odds of default by 1.8". Regulators, auditors, and declined customers can all be given a reason.

The transferable skills here are exactly those of a scorecard build:

- **Prepare mixed, missing data cleanly** (impute, encode) inside a leakage-safe pipeline.
- **Turn coefficients into odds ratios** so the model tells a story, not just a number.
- **Choose the decision threshold on purpose**, trading false alarms against missed events according to their business cost, rather than blindly using 0.5.

## Index

1. [Setup](#setup)
2. [Get the data](#data)
3. [Exploratory data analysis](#eda)
4. [Prepare the data (missing values and feature choice)](#prep)
5. [Train / test split](#split)
6. [Build and fit a logistic regression pipeline](#model)
7. [Evaluate: confusion matrix, report, ROC and AUC](#eval)
8. [Interpret the coefficients as odds ratios](#odds)
9. [Move the decision threshold](#threshold)
10. [Further practice](#further)
11. [Key takeaways](#takeaways)

<a id="setup"></a>
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, accuracy_score)

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")

<a id="data"></a>
## 2. Get the data

### Exercise 1: Load the data

Read `../../datasets/train_titanic.csv` into a DataFrame called `titanic`. Show `titanic.head()`.

In [ ]:
# Your turn


In [ ]:
# Coach answer
# Or read directly from the public S3 bucket (no local file needed):
# titanic = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/train_titanic.csv")
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# titanic = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/train_titanic.csv", header=True, inferSchema=True).toPandas()
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets
# titanic = spark.read.csv(session_datasets["train_titanic"], header=True, inferSchema=True).toPandas()
titanic = pd.read_csv("../../datasets/Session_5/train_titanic.csv")
titanic.head()

### Exercise 2: Inspect the data and missing values

Call `titanic.info()`, print the survival balance, and print the count of missing values per column (`isna().sum()`). Which columns have serious missingness?

In [ ]:
# Your turn


In [ ]:
# Coach answer
titanic.info()
print()
print("Survival balance:")
print(titanic["Survived"].value_counts())
print("Survival rate:", round(titanic["Survived"].mean(), 3))
print()
print("Missing values per column:")
print(titanic.isna().sum())
# Age is missing for 177 (impute it), Cabin for most (drop it), Embarked for 2 (impute it).

<a id="eda"></a>
## 3. Exploratory data analysis

A couple of plots to see which passengers survived.

### Exercise 3: Survival by sex and by class

Make two countplots side by side (or one at a time): survival counts split by `Sex`, and survival counts split by `Pclass`. Who was more likely to survive?

> *Hint:* `sns.countplot(data=titanic, x="Sex", hue="Survived")`.

In [ ]:
# Your turn


In [ ]:
# Coach answer
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=titanic, x="Sex", hue="Survived", ax=axes[0])
axes[0].set_title("Survival by sex")
sns.countplot(data=titanic, x="Pclass", hue="Survived", ax=axes[1])
axes[1].set_title("Survival by passenger class")
plt.tight_layout()
plt.show()
# Females survived at a much higher rate than males; first-class survived far more than third.

### Exercise 4: Age distribution

Draw a histogram of `Age` (dropping missing values). What is the rough age profile of the passengers?

> *Hint:* `titanic["Age"].dropna().hist(bins=30)`.

In [ ]:
# Your turn


In [ ]:
# Coach answer
plt.figure(figsize=(9, 4))
titanic["Age"].dropna().hist(bins=30, color="steelblue")
plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Passenger age distribution")
plt.show()

<a id="prep"></a>
## 4. Prepare the data (missing values and feature choice)

We keep a sensible set of predictors and drop columns that are mostly missing or that are identifiers with no predictive value.

- **Drop** `PassengerId`, `Name`, `Ticket` (identifiers/text) and `Cabin` (687 of 891 missing).
- **Numeric** predictors: `Age`, `Fare`, `SibSp`, `Parch`. `Age` needs imputing.
- **Categorical** predictors: `Sex`, `Embarked`, `Pclass`. `Embarked` needs imputing.

Rather than clean columns by hand, we will let a `ColumnTransformer` inside a pipeline do the imputing and encoding, so it is fitted on the training data only (no leakage).

### Exercise 5: Choose features and target

Build `X` with the predictor columns `["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]` and `y` as `Survived`. Define two lists, `num_feats` and `cat_feats`, naming the numeric and categorical predictors (treat `Pclass` as categorical).

In [ ]:
# Your turn


In [ ]:
# Coach answer
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = titanic[features].copy()
y = titanic["Survived"]

num_feats = ["Age", "Fare", "SibSp", "Parch"]
cat_feats = ["Pclass", "Sex", "Embarked"]
print("Numeric:", num_feats)
print("Categorical:", cat_feats)

<a id="split"></a>
## 5. Train / test split

### Exercise 6: Stratified train/test split

Split into train/test with `test_size=0.30`, `random_state=RANDOM_STATE`, and `stratify=y` so both sets keep the survival rate.

In [ ]:
# Your turn


In [ ]:
# Coach answer
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Survival rate  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

<a id="model"></a>
## 6. Build and fit a logistic regression pipeline

We assemble a `ColumnTransformer` that:

- imputes missing `Age`/`Fare` with the median and standardises the numeric features, and
- imputes missing `Embarked` with the most frequent value and one-hot encodes the categorical features (`drop="first"` to avoid the dummy trap).

Then we chain it with a `LogisticRegression` in one pipeline.

### Exercise 7: Build the preprocessing + model pipeline

Build the numeric pipeline (`SimpleImputer(strategy="median")` then `StandardScaler()`), the categorical pipeline (`SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(drop="first", handle_unknown="ignore")`), combine them in a `ColumnTransformer` called `prep`, and put `prep` + `LogisticRegression(max_iter=1000)` into a pipeline called `logreg`. Fit it on the training data.

In [ ]:
# Your turn


In [ ]:
# Coach answer
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
])
prep = ColumnTransformer([
    ("num", num_pipe, num_feats),
    ("cat", cat_pipe, cat_feats),
])

logreg = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000)),
])
logreg.fit(X_train, y_train)
print("Fitted logistic regression pipeline.")

<a id="eval"></a>
## 7. Evaluate: confusion matrix, report, ROC and AUC

### Exercise 8: Confusion matrix and classification report

Predict on `X_test` (call it `pred`), then print the accuracy, the confusion matrix, and the classification report.

In [ ]:
# Your turn


In [ ]:
# Coach answer
pred = logreg.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, target_names=["died (0)", "survived (1)"]))

### Exercise 9: ROC curve and AUC

Get the predicted probabilities of survival with `logreg.predict_proba(X_test)[:, 1]`, compute the AUC with `roc_auc_score`, and plot the ROC curve (`roc_curve`). Add the diagonal "random" reference line.

> *Hint:* AUC summarises the ROC curve in one number; 0.5 is random, 1.0 is perfect.

In [ ]:
# Your turn


In [ ]:
# Coach answer
proba = logreg.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
fpr, tpr, thresholds = roc_curve(y_test, proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Logistic regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
plt.show()
print("AUC:", round(auc, 3))

<a id="odds"></a>
## 8. Interpret the coefficients as odds ratios

This is what logistic regression is prized for. Each coefficient is on the **log-odds** scale; exponentiating it (`exp(coef)`) turns it into an **odds ratio**:

- odds ratio **> 1**: the feature **increases** the odds of survival,
- odds ratio **< 1**: the feature **decreases** them,
- odds ratio **= 1**: no effect.

Because we standardised the numeric features, their odds ratios are "per one standard deviation increase". The categorical odds ratios compare each category to the dropped baseline (for example `Sex_male` compares males to females).

### Exercise 10: Build an odds-ratio table

Pull the fitted feature names from the preprocessing step (`logreg.named_steps["prep"].get_feature_names_out()`) and the coefficients from the classifier (`logreg.named_steps["clf"].coef_[0]`). Build a DataFrame with the coefficient and its odds ratio (`np.exp(coef)`), sorted by odds ratio. Which feature most reduces the odds of survival?

In [ ]:
# Your turn


In [ ]:
# Coach answer
names = logreg.named_steps["prep"].get_feature_names_out()
coefs = logreg.named_steps["clf"].coef_[0]

odds = pd.DataFrame({"feature": names, "coefficient": coefs})
odds["odds_ratio"] = np.exp(odds["coefficient"])
odds = odds.sort_values("odds_ratio")
print(odds.to_string(index=False))
# 'cat__Sex_male' has by far the smallest odds ratio: being male (vs female) multiplies the odds of
# survival by a small factor, i.e. it drastically lowers them. Higher Pclass (3rd vs 1st) also cuts
# the odds sharply, while higher Fare nudges them up. This matches 'women and first class first'.

### Exercise 11: Say one odds ratio in plain English

Take the `Sex_male` odds ratio from the table and write a one-sentence, plain-English interpretation of it (as you would for a stakeholder). Print your sentence.

In [ ]:
# Your turn


In [ ]:
# Coach answer
or_male = odds.loc[odds["feature"] == "cat__Sex_male", "odds_ratio"].values[0]
print(f"Odds ratio for being male: {or_male:.3f}")
print(f"Interpretation: holding the other features constant, a male passenger's odds of survival "
      f"were about {or_male:.2f} times those of a female passenger, i.e. roughly "
      f"{(1 - or_male) * 100:.0f}% lower.")

<a id="threshold"></a>
## 9. Move the decision threshold

By default the model predicts "survived" when the predicted probability exceeds 0.5. That cut-off is a choice, not a law. Raising or lowering it trades **precision** against **recall**. In a risk setting you would set it by the relative cost of a false alarm versus a missed event.

### Exercise 12: Compare thresholds

Using the `proba` from Exercise 9, relabel predictions at thresholds 0.5, 0.4, and 0.3. For each, print the recall and precision on the survived class (class 1). What happens as you lower the threshold?

> *Hint:* `from sklearn.metrics import recall_score, precision_score`; `(proba >= t).astype(int)`.

In [ ]:
# Your turn


In [ ]:
# Coach answer
from sklearn.metrics import recall_score, precision_score
for t in [0.5, 0.4, 0.3]:
    p = (proba >= t).astype(int)
    print(f"threshold={t}:  recall(survived)={recall_score(y_test, p):.3f}"
          f"   precision(survived)={precision_score(y_test, p):.3f}"
          f"   accuracy={accuracy_score(y_test, p):.3f}")
# Lowering the threshold catches more true survivors (recall up) but at the cost of more false
# positives (precision down). The 'right' threshold depends on how costly each error type is.

<a id="further"></a>
## 10. Further practice

1. **Feature engineering.** Create `FamilySize = SibSp + Parch + 1` and an `IsAlone` flag, add them, and see if AUC improves.
2. **Regularisation.** Logistic regression has a `C` parameter (inverse regularisation strength). Grid-search it with `GridSearchCV` and `scoring="roc_auc"`.
3. **Compare a model.** Fit a `RandomForestClassifier` on the same pipeline preprocessing and compare AUC. Does the more flexible model beat the interpretable one here?

### Exercise 13 (stretch): add a FamilySize feature

Add `FamilySize = SibSp + Parch + 1` to `X_train` and `X_test`, include it in `num_feats`, refit the pipeline, and compare the new AUC to the old one.

> *Hint:* rebuild the `ColumnTransformer` and pipeline after adding the feature to `num_feats`.

In [ ]:
# Your turn


In [ ]:
# Coach answer
X_train2 = X_train.copy()
X_test2 = X_test.copy()
X_train2["FamilySize"] = X_train2["SibSp"] + X_train2["Parch"] + 1
X_test2["FamilySize"] = X_test2["SibSp"] + X_test2["Parch"] + 1

num_feats2 = num_feats + ["FamilySize"]
prep2 = ColumnTransformer([
    ("num", num_pipe, num_feats2),
    ("cat", cat_pipe, cat_feats),
])
logreg2 = Pipeline([("prep", prep2), ("clf", LogisticRegression(max_iter=1000))])
logreg2.fit(X_train2, y_train)
auc2 = roc_auc_score(y_test, logreg2.predict_proba(X_test2)[:, 1])
print(f"AUC without FamilySize: {auc:.3f}")
print(f"AUC with FamilySize:    {auc2:.3f}")
# A small, engineered feature can nudge performance; here the AUC is essentially unchanged because
# SibSp and Parch already carry that signal, but this shows the workflow for adding features cleanly.
# Not every engineered feature helps, and checking (rather than assuming) is the point.

<a id="takeaways"></a>
## 11. Key takeaways

| Step | What you did | Why it matters |
|---|---|---|
| Clean in a pipeline | Impute + encode via `ColumnTransformer` | Handles missing/mixed data with no leakage from the test set |
| Evaluate broadly | Confusion matrix, report, ROC/AUC | Accuracy alone misses how well probabilities rank cases |
| Odds ratios | `exp(coefficient)` | Turns the model into plain-language, auditable statements |
| Move the threshold | `predict_proba` + chosen cut-off | Set the false-alarm vs missed-event balance to the business cost |

**The one-line lesson:** logistic regression earns its place in risk work by being explainable. Prepare the data cleanly, read it through odds ratios and the ROC curve, and choose the threshold deliberately.

Great job. You have built and, crucially, explained a logistic regression model end to end.